# Chatbot Expert AI Act — Routage deterministe + memoire conversationnelle

3 modes automatiques decides par du code Python :
1. **DIRECT** : regex article/considerant/annexe → texte mot a mot (0 LLM), ou analyse LLM si resume demande
2. **RAG** : FAISS pertinent → LLM redige (1 LLM)
3. **WEB** : rien dans FAISS → DuckDuckGo + LLM (1 LLM)

**Multi-articles** : "articles 5 et 8", "articles 5 a 8", "considerants 1 et 2", "annexe III"

**2 environnements supportes** :
- **Local** : Ollama (Qwen 2.5 3B) — detection automatique
- **Cloud** : Groq (Llama 3.1 8B Instant) — cle GROQ_API_KEY gratuite

---
## Cellule 1 — Installation des dépendances

In [56]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])

0

---
## Cellule 2 — Imports

In [ ]:
import os
import re
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_community.tools import DuckDuckGoSearchRun

print("Imports OK")

---
## Cellule 3 — Configuration
Modifiez les chemins si nécessaire.

In [ ]:
# --- CONFIGURATION ---
MD_FILE          = Path("OJ_L_202401689_FR_TXTavec annexes.md")
INDEX_DIR        = Path("faiss_index")
MODEL_NAME       = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
TOP_K            = 8
SCORE_THRESHOLD  = 0.35

# Modeles LLM
OLLAMA_MODEL     = "qwen2.5:3b"              # Local (Ollama)
GROQ_MODEL       = "llama-3.1-8b-instant"    # Cloud (Groq, gratuit)

print(f"Fichier source : {MD_FILE} ({'existe' if MD_FILE.exists() else 'INTROUVABLE'})")
print(f"Index FAISS    : {INDEX_DIR} ({'existe' if INDEX_DIR.exists() else 'a construire'})")

---
## Cellule 4 — Fonctions de parsing et chunking

Le texte de l'AI Act est un reglement europeen avec une structure tres hierarchisee :
- **Considerants** (1) a (180+) : motivations du texte
- **13 Chapitres** > **Sections** > **113 Articles** numerotes
- **13 Annexes** (I a XIII) : listes, criteres, documentation technique

La strategie de chunking exploite cette structure : **1 chunk = 1 considerant, 1 article ou 1 section d'annexe** (decoupe par paragraphe si trop long), avec un prefixe hierarchique et des metadonnees riches.

In [ ]:
def clean_text(text: str) -> str:
    """Nettoie les artefacts Markdown : separateurs de tableau, liens, pages."""
    text = re.sub(r"\|---\|---\|", "", text)
    text = re.sub(r"\|([a-z]\))\|(.+?)\|", r"\1 \2", text)
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)
    # Supprimer les marqueurs de page PDF (ex: 44/144   ELI: http://...)
    text = re.sub(r"\d+/\d+\s+ELI:\s+http://\S+", "", text)
    # Supprimer les en-tetes de page JO
    text = re.sub(r"JO L du \d+\.\d+\.\d+\s+FR", "", text)
    # Supprimer les ## restants (titres markdown)
    text = re.sub(r"^##\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def parse_ai_act(filepath: Path = MD_FILE) -> list[dict]:
    """
    Parse le fichier Markdown de l'AI Act et retourne une liste de chunks.
    Supporte le format PDF->MD (## CHAPITRE, ## Article, (N) considerants).
    Inclut les 13 annexes (I a XIII).
    """
    text = Path(filepath).read_text(encoding="utf-8")
    text = re.sub(r"^---.*?---", "", text, count=1, flags=re.DOTALL)
    text = re.sub(r"^#\s+.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^>\s+.*$", "", text, flags=re.MULTILINE)
    text = text.replace("\xa0", " ")

    lines = text.split("\n")
    chunks = []

    # ========== Phase 1 : Considerants (avant CHAPITRE I) ==========
    chapitre_start = None
    for idx, line in enumerate(lines):
        stripped = line.strip().replace("##", "").strip()
        if stripped == "CHAPITRE I":
            chapitre_start = idx
            break

    if chapitre_start:
        current_num = None
        current_body = []

        def flush_considerant():
            if current_num and current_body:
                body = " ".join(current_body)
                cleaned = clean_text(body)
                if len(cleaned) >= 20:
                    chunks.append({
                        "content": f"Considerant {current_num} : {cleaned}",
                        "metadata": {
                            "type": "considerant", "numero": current_num,
                            "chapter": "", "section": "", "article": "",
                            "title": f"Considerant {current_num}",
                        },
                    })

        for idx in range(chapitre_start):
            line = lines[idx]
            # Format ancien : |(N)|texte|
            m_old = re.match(r"^\|(\(\d+\))\|(.+)\|$", line)
            if m_old:
                flush_considerant()
                current_num = m_old.group(1)
                current_body = [m_old.group(2)]
                continue
            # Format nouveau : (N) texte
            m_new = re.match(r"^\((\d+)\)\s+(.+)", line.strip())
            if m_new:
                flush_considerant()
                current_num = f"({m_new.group(1)})"
                current_body = [m_new.group(2)]
                continue
            stripped = line.strip()
            if stripped and current_num and not stripped.startswith("#"):
                current_body.append(stripped)
        flush_considerant()

    # ========== Phase 2 : Articles (apres CHAPITRE I, avant ANNEXE I) ==========
    current_chapter = ""
    current_chapter_title = ""
    current_section = ""
    current_section_title = ""

    annexe_start = len(lines)
    for idx, line in enumerate(lines):
        if re.match(r"^(##\s+)?ANNEXE\s+[IVXLC]+\s*$", line.strip()):
            annexe_start = idx
            break

    i = chapitre_start or 0
    while i < annexe_start:
        line = lines[i].strip()
        bare = line.replace("##", "").strip()

        match_chap = re.match(r"^CHAPITRE\s+([IVXLC]+)$", bare)
        if match_chap:
            current_chapter = match_chap.group(1)
            current_section = ""
            current_section_title = ""
            j = i + 1
            while j < annexe_start and lines[j].strip().replace("##", "").strip() == "":
                j += 1
            if j < annexe_start:
                current_chapter_title = lines[j].strip().replace("##", "").strip()
            i = j + 1
            continue

        match_sec = re.match(r"^SECTION\s+(\d+)$", bare)
        if match_sec:
            current_section = match_sec.group(1)
            j = i + 1
            while j < annexe_start and lines[j].strip().replace("##", "").strip() == "":
                j += 1
            if j < annexe_start:
                current_section_title = lines[j].strip().replace("##", "").strip()
            i = j + 1
            continue

        match_art = re.match(r"^Article\s+(premier|\d+)$", bare)
        if match_art:
            art_num = match_art.group(1)
            if art_num == "premier":
                art_num = "1"
            j = i + 1
            while j < annexe_start and lines[j].strip().replace("##", "").strip() == "":
                j += 1
            art_title = lines[j].strip().replace("##", "").strip() if j < annexe_start else ""
            j += 1
            content_lines = []
            while j < annexe_start:
                next_bare = lines[j].strip().replace("##", "").strip()
                if re.match(r"^(Article\s+(premier|\d+)|CHAPITRE\s+[IVXLC]+|SECTION\s+\d+)$", next_bare):
                    break
                content_lines.append(lines[j])
                j += 1
            raw_content = "\n".join(content_lines)
            cleaned = clean_text(raw_content)
            if len(cleaned) < 10:
                i = j
                continue
            prefix_parts = []
            if current_chapter:
                prefix_parts.append(f"Chapitre {current_chapter} - {current_chapter_title}")
            if current_section:
                prefix_parts.append(f"Section {current_section} - {current_section_title}")
            prefix_parts.append(f"Article {art_num} : {art_title}")
            prefix = " > ".join(prefix_parts)
            full_content = f"{prefix}\n\n{cleaned}"
            if len(full_content) > 2000:
                paragraphs = re.split(r"\n(?=\d+\.\s{2,})", cleaned)
                for p_idx, para in enumerate(paragraphs):
                    para = para.strip()
                    if len(para) < 20:
                        continue
                    chunks.append({
                        "content": f"{prefix}\n\n{para}",
                        "metadata": {
                            "type": "article", "chapter": current_chapter,
                            "chapter_title": current_chapter_title,
                            "section": current_section, "section_title": current_section_title,
                            "article": art_num, "title": art_title,
                            "paragraph": str(p_idx + 1),
                        },
                    })
            else:
                chunks.append({
                    "content": full_content,
                    "metadata": {
                        "type": "article", "chapter": current_chapter,
                        "chapter_title": current_chapter_title,
                        "section": current_section, "section_title": current_section_title,
                        "article": art_num, "title": art_title, "paragraph": "",
                    },
                })
            i = j
            continue
        i += 1

    # ========== Phase 3 : Annexes ==========
    i = annexe_start
    while i < len(lines):
        line = lines[i].strip()
        bare = line.replace("##", "").strip()
        match_annexe = re.match(r"^ANNEXE\s+([IVXLC]+)$", bare)
        if match_annexe:
            annexe_num = match_annexe.group(1)
            j = i + 1
            while j < len(lines) and lines[j].strip().replace("##", "").strip() == "":
                j += 1
            annexe_title = lines[j].strip().replace("##", "").strip() if j < len(lines) else ""
            j += 1
            content_lines = []
            while j < len(lines):
                next_bare = lines[j].strip().replace("##", "").strip()
                if re.match(r"^ANNEXE\s+[IVXLC]+$", next_bare):
                    break
                content_lines.append(lines[j])
                j += 1
            raw_content = "\n".join(content_lines)
            cleaned = clean_text(raw_content)
            if len(cleaned) < 10:
                i = j
                continue
            prefix = f"Annexe {annexe_num} : {annexe_title}"
            full_content = f"{prefix}\n\n{cleaned}"
            if len(full_content) > 2000:
                sections = re.split(r"\n(?=\d+\.\s{2,}|\d+\.\s+[A-Z])", cleaned)
                for s_idx, section in enumerate(sections):
                    section = section.strip()
                    if len(section) < 20:
                        continue
                    chunks.append({
                        "content": f"{prefix}\n\n{section}",
                        "metadata": {
                            "type": "annexe", "annexe": annexe_num,
                            "title": annexe_title, "section": str(s_idx + 1),
                            "chapter": "", "article": "",
                        },
                    })
            else:
                chunks.append({
                    "content": full_content,
                    "metadata": {
                        "type": "annexe", "annexe": annexe_num,
                        "title": annexe_title, "section": "",
                        "chapter": "", "article": "",
                    },
                })
            i = j
            continue
        i += 1

    return chunks

print("Fonctions de parsing definies (articles + considerants + annexes)")

---
## Cellule 5 — Parsing du AI Act
On exécute le chunker et on affiche les statistiques.

In [ ]:
chunks = parse_ai_act()

considerants = [c for c in chunks if c["metadata"]["type"] == "considerant"]
articles     = [c for c in chunks if c["metadata"]["type"] == "article"]
annexes      = [c for c in chunks if c["metadata"]["type"] == "annexe"]

print(f"Nombre total de chunks : {len(chunks)}")
print(f"  - Considerants : {len(considerants)}")
print(f"  - Articles     : {len(articles)}")
print(f"  - Annexes      : {len(annexes)}")
print()

annexe_nums = sorted(set(c["metadata"]["annexe"] for c in annexes),
                     key=lambda x: len(x) * 100 + ord(x[0]))
print(f"Annexes trouvees : {', '.join(annexe_nums)}")
print()

for c in chunks[:2] + [c for c in chunks if c["metadata"]["type"] == "annexe"][:1]:
    print(f"[{c['metadata']['type']}] {c['metadata'].get('title', '')}")
    print(c["content"][:150] + "...")
    print()

---
## Cellule 6 — Construction de l'index FAISS

On encode les 641 chunks avec `paraphrase-multilingual-mpnet-base-v2` (278M paramètres, 768 dimensions) et on les indexe dans FAISS.

**⏱ Première exécution** : ~2-3 min (téléchargement du modèle ~1 Go + encodage).  
**Exécutions suivantes** : ~30s (modèle en cache).

Si l'index existe déjà, cette cellule le recharge simplement.

In [61]:
# Charger le modèle d'embeddings
print(f"Chargement du modèle : {MODEL_NAME}")
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu"},              # "cuda" si GPU disponible
    encode_kwargs={
        "normalize_embeddings": True,             # Norme L2=1 → cosine similarity = dot product
        "batch_size": 32,                         # Encode 32 textes à la fois
    },
)
print("Modèle chargé ✓")

# Construire ou recharger l'index
if INDEX_DIR.exists():
    print("Index FAISS existant trouvé → rechargement...")
    db = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
else:
    print("Construction de l'index FAISS...")
    documents = [
        Document(page_content=c["content"], metadata=c["metadata"])
        for c in chunks
    ]
    db = FAISS.from_documents(documents, embeddings)
    db.save_local(str(INDEX_DIR))
    print(f"Index sauvegardé dans {INDEX_DIR}/")

print(f"Index FAISS prêt ✓ ({db.index.ntotal} vecteurs de dimension {db.index.d})")

Chargement du modèle : sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5847.42it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèle chargé ✓
Index FAISS existant trouvé → rechargement...
Index FAISS prêt ✓ (641 vecteurs de dimension 768)


---
## Cellule 7 — Test de la recherche vectorielle (sans LLM)
Vérifions que FAISS retrouve les bons articles avant de brancher le LLM.

In [62]:
question_test = "Quelles sont les pratiques d'IA interdites ?"

docs = db.similarity_search(question_test, k=TOP_K)

print(f"Question : {question_test}\n")
print(f"{len(docs)} documents retrouvés :\n")
for i, doc in enumerate(docs):
    m = doc.metadata
    label = m.get("title", "")
    if m.get("article"):
        label = f"Article {m['article']} : {label}"
    if m.get("chapter"):
        label = f"Chap. {m['chapter']} > {label}"
    print(f"  {i+1}. [{m['type']}] {label}")
    print(f"     {doc.page_content[:120]}...\n")

Question : Quelles sont les pratiques d'IA interdites ?

5 documents retrouvés :

  1. [article] Chap. II > Article 5 : Pratiques interdites en matière d’IA
     Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

1.   Les pratique...

  2. [article] Chap. II > Article 5 : Pratiques interdites en matière d’IA
     Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

8.   Le présent a...

  3. [article] Chap. II > Article 5 : Pratiques interdites en matière d’IA
     Chapitre II - PRATIQUES INTERDITES EN MATIÈRE D’IA > Article 5 : Pratiques interdites en matière d’IA

2.   L’utilisatio...

  4. [considerant] Considérant (28)
     Considérant (28) : Si l’IA peut être utilisée à de nombreuses fins positives, elle peut aussi être utilisée à mauvais es...

  5. [considerant] Considérant (132)
     Considérant (132) : Certains systèmes d’IA destinés à interagir avec des personnes physiques ou

---
## Cellule 8 — Connexion LLM (detection automatique Local/Cloud)

- **Local** : si Ollama repond sur localhost → `ChatOllama`
- **Cloud** : sinon → `ChatGroq` (cle `GROQ_API_KEY` requise, gratuite sur https://console.groq.com)

In [ ]:
import requests as _req

def is_ollama_available():
    try:
        return _req.get("http://localhost:11434/api/tags", timeout=2).status_code == 200
    except Exception:
        return False

if is_ollama_available():
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.1)
    LLM_LABEL = f"Ollama ({OLLAMA_MODEL})"
else:
    from langchain_groq import ChatGroq
    groq_api_key = os.environ.get("GROQ_API_KEY", "")
    if not groq_api_key:
        raise ValueError(
            "Cle GROQ_API_KEY requise.\n"
            "1. Creez un compte sur https://console.groq.com\n"
            "2. Generez une cle API\n"
            "3. os.environ['GROQ_API_KEY'] = 'gsk_...'"
        )
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0.1)
    LLM_LABEL = f"Groq ({GROQ_MODEL})"

print(f"LLM : {LLM_LABEL}")

# Test rapide
try:
    test = llm.invoke("Reponds OK.")
    content = test.content if hasattr(test, 'content') else str(test)
    print(f"Test : {content[:50]}")
except Exception as e:
    print(f"ERREUR : {e}")

---
## Cellule 9 — Fonctions de base + Retriever + Prompt + Memoire conditionnelle

In [ ]:
# ====== Fonctions de base ======

def docstore_lookup(db, filters):
    results = []
    for doc_id in db.index_to_docstore_id.values():
        doc = db.docstore.search(doc_id)
        if all(doc.metadata.get(k) == v for k, v in filters.items()):
            results.append(doc)
    results.sort(key=lambda d: int(d.metadata.get("paragraph", "0") or "0"))
    return results

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

def get_sources(docs):
    sources = []
    for doc in docs:
        m = doc.metadata
        if m.get("type") == "article":
            label = f"Article {m['article']} : {m['title']}"
            if m.get("chapter"):
                label = f"Chapitre {m['chapter']} > {label}"
        elif m.get("type") == "annexe":
            label = f"Annexe {m['annexe']} : {m['title']}"
        else:
            label = m.get("title", "Considerant")
        sources.append(label)
    return sources

# ====== Retriever ======
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": TOP_K, "score_threshold": SCORE_THRESHOLD},
)

# ====== Prompt ameliore ======
RESPONSE_PROMPT = """\
Tu es un assistant expert du Reglement europeen sur l'Intelligence Artificielle \
(AI Act, Reglement UE 2024/1689). Reponds en francais.

REGLES :
1. Base ta reponse sur le contexte fourni ET sur l'historique de conversation.
2. Quand le contexte vient du AI Act, cite les articles et considerants exacts \
   (ex: "Article 6, paragraphe 2").
3. Si le contexte contient des obligations ou interdictions, LISTE-LES precisement.
4. Si le contexte vient d'internet, PRECISE-LE clairement et reponds normalement \
   sans forcer de lien avec le AI Act.
5. Ne dis JAMAIS "consultez le texte complet" ou "je n'ai pas assez d'info".
   Utilise ce que tu as et reponds du mieux possible.
6. Structure ta reponse avec des titres et des puces si necessaire.
7. Tu as acces a l'historique de conversation. Si l'utilisateur fait reference \
   a un echange precedent (un prenom, un sujet, une personne), utilise l'historique \
   pour repondre. Ne dis JAMAIS "je ne sais pas" si l'info est dans l'historique.
8. Si l'utilisateur demande un RESUME ou une EXPLICATION d'un article ou considerant, \
   fournis un resume synthetique, pas le texte integral.
9. Pour les questions sans rapport avec le AI Act (blagues, culture generale, etc.), \
   reponds normalement sans forcer de lien avec le reglement.
"""

# ====== Memoire systematique ======
chat_history = InMemoryChatMessageHistory()

def call_llm(question, context, context_label="AI Act"):
    """Appel LLM avec historique TOUJOURS inclus."""
    messages = [SystemMessage(content=RESPONSE_PROMPT)]
    # Toujours inclure l'historique
    if chat_history.messages:
        for msg in chat_history.messages[-6:]:
            content = msg.content[:800] + "..." if len(msg.content) > 800 else msg.content
            messages.append(type(msg)(content=content))
    messages.append(HumanMessage(
        content=f"Contexte ({context_label}) :\n{context}\n\nQuestion : {question}"
    ))
    return llm.invoke(messages).content

print("Fonctions, retriever, prompt, memoire prets")

---
## Cellule 10 — Outils de recherche

In [ ]:
def recherche_article(article_num):
    docs = docstore_lookup(db, {"article": article_num})
    if not docs: return f"Article {article_num} non trouve."
    return format_docs(docs)

def recherche_considerant(numero):
    docs = docstore_lookup(db, {"type": "considerant", "numero": f"({numero})"})
    if not docs: return f"Considerant {numero} non trouve."
    return docs[0].page_content

def recherche_ia_act(query):
    """Retourne (contexte_avec_sources, liste_sources)."""
    docs = retriever.invoke(query)
    if not docs: return "", []
    sources = get_sources(docs)
    header = "Sources AI Act :\n" + "\n".join(f"- {s}" for s in sources) + "\n\nExtraits :\n"
    return header + format_docs(docs), sources

def recherche_web(query):
    search = DuckDuckGoSearchRun()
    parts = []
    try: parts.append(search.invoke(query))
    except Exception: pass
    try: parts.append(search.invoke(query + " results 2025"))
    except Exception: pass
    return "\n\n".join(p for p in parts if p)

print("4 outils prets")

---
## Cellule 11 — Fonction ask() : routage deterministe + memoire

In [ ]:
def _direct_or_llm(docs, question, wants_analysis, label):
    """Texte brut si simple demande, LLM si resume/explication demande."""
    sources = get_sources(docs)
    context = format_docs(docs)
    if wants_analysis:
        if len(context) > 4000:
            context = context[:4000] + "\n\n[... tronque ...]"
        response = call_llm(question, context, f"AI Act - {label}")
        print(f"[DIRECT+LLM] {label} (analyse)\n" + "-" * 70)
        print(response)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=response))
        return response
    print(f"[DIRECT] {label} (texte integral)\n" + "-" * 70)
    print(context)
    chat_history.add_message(HumanMessage(content=question))
    chat_history.add_message(AIMessage(content=context[:500]))
    return context


def ask(question):
    print(f"\nQuestion : {question}")
    print("=" * 70)
    q = question.lower()

    wants_analysis = bool(re.search(
        r"(r[eé]sum|synth[eé]|explique|compar|analys|simplifie|vulgar|tradui|"
        r"d[eé]taille|d[eé]veloppe|pr[eé]cise|reformule|
        r"en quoi|que signifie|que veut dire|qu.?est.ce que|comment)"",
        q
    ))

    # === MODE 1 : DIRECT ===
    range_match = re.search(r"articles?\s+(\d+)\s+[\u00e0a]\s+(\d+)", q)
    if range_match:
        start, end = int(range_match.group(1)), int(range_match.group(2))
        all_docs, nums = [], []
        for n in range(start, min(end + 1, start + 20)):
            docs = docstore_lookup(db, {"article": str(n)})
            if docs: all_docs.extend(docs); nums.append(str(n))
        if all_docs:
            return _direct_or_llm(all_docs, question, wants_analysis, f"Articles {', '.join(nums)}")

    art_block = re.search(r"(?:articles?|art\.?)\s+((?:premier|\d+)(?:\s*(?:,|et)\s*(?:premier|\d+))*)", q)
    art_matches = re.findall(r"(premier|\d+)", art_block.group(1)) if art_block else []
    if art_matches:
        all_docs, nums = [], []
        for m in art_matches:
            num = "1" if m == "premier" else m
            if num not in nums: nums.append(num); all_docs.extend(docstore_lookup(db, {"article": num}))
        if all_docs:
            return _direct_or_llm(all_docs, question, wants_analysis, f"Article{'s' if len(nums)>1 else ''} {', '.join(nums)}")

    cons_block = re.search(r"consid[\u00e9e]rants?\s+((?:\d+)(?:\s*(?:,|et)\s*\d+)*)", q)
    cons_matches = re.findall(r"(\d+)", cons_block.group(1)) if cons_block else []
    if cons_matches:
        all_docs = []
        for num in cons_matches: all_docs.extend(docstore_lookup(db, {"type": "considerant", "numero": f"({num})"}))
        if all_docs:
            return _direct_or_llm(all_docs, question, wants_analysis, f"Considerant{'s' if len(cons_matches)>1 else ''} {', '.join(cons_matches)}")

    annexe_matches = re.findall(r"annexe\s+([IVXLC]+|\d+)", q, re.IGNORECASE)
    if annexe_matches:
        all_docs, nums = [], []
        for num in annexe_matches:
            u = num.upper()
            if u not in nums: nums.append(u); all_docs.extend(docstore_lookup(db, {"type": "annexe", "annexe": u}))
        if all_docs:
            return _direct_or_llm(all_docs, question, wants_analysis, f"Annexe{'s' if len(nums)>1 else ''} {', '.join(nums)}")

    # === MODE 2 : RAG ===
    docs = retriever.invoke(question)
    if docs:
        sources = get_sources(docs)
        context = format_docs(docs)
        source_list = "\n".join(f"- {s}" for s in sources)
        if len(context) > 4000: context = context[:4000] + "\n\n[... tronque ...]"
        response = call_llm(question, f"Sources AI Act :\n{source_list}\n\nExtraits :\n{context}", "AI Act")
        print(f"[RAG] {len(sources)} docs : {sources[:4]}\n" + "-" * 70)
        print(response)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=response))
        return response

    # === MODE 2bis : CONVERSATION (suivi avec pronoms) ===
    if chat_history.messages:
        is_short = len(question.split()) < 15
        has_pronoun = bool(re.search(
            r"\b(il|elle|ils|elles|lui|son|sa|ses|leur|ce|cet|cette|ces|"
            r"le meme|la meme|aussi|egalement|en plus|de plus|"
            r"je m.appelle|mon nom|comment je|qui suis|quel |sur quel|"
            r"a.?t.?il|a.?t.?elle|est.?il|est.?elle|etait.?il|etait.?elle|"
            r"le pr[\u00e9e]c[\u00e9e]dent|ci.?dessus|plus haut|ta r[\u00e9e]ponse)\b", q))
        action_followup = bool(re.match(
            r"^(r[eé]sum|synth[eé]|explique|d[eé]taille|d[eé]veloppe|pr[eé]cise|"
            r"reformule|tradui|simplifie|continue|poursui|compl[eè]te|approfondi|"
            r"r[eé]p[eè]te|redis|relis)",
            q
        ))
        if is_short and (has_pronoun or action_followup):
            response = call_llm(question, "Pas de contexte. Reponds avec l'historique.", "Conversation")
            print(f"[CONVERSATION] Memoire\n" + "-" * 70); print(response)
            chat_history.add_message(HumanMessage(content=question))
            chat_history.add_message(AIMessage(content=response))
            return response

    # === MODE 3 : LLM SEUL (connaissances propres) ===
    print("[LLM] Connaissances propres...")
    response = call_llm(question,
        "Aucun document dans la base AI Act. Reponds avec tes propres connaissances. "
        "Si tu ne connais PAS la reponse, reponds EXACTEMENT [RECHERCHE_WEB] sur la premiere ligne.",
        "Connaissances du modele")
    if "[RECHERCHE_WEB]" not in response:
        print(f"[LLM] Connaissances propres\n" + "-" * 70); print(response)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=response))
        return response

    # === MODE 4 : WEB (DuckDuckGo) ===
    print("[WEB] Le LLM ne sait pas, recherche internet...")
    search = DuckDuckGoSearchRun()
    web_parts = []
    try: web_parts.append(search.invoke(question))
    except: pass
    try: web_parts.append(search.invoke(question + " results 2025"))
    except: pass
    web_results = "\n\n".join(p for p in web_parts if p)
    if web_results and len(web_results) > 50:
        response = call_llm(question, web_results[:2500], "Internet (DuckDuckGo)")
        print(f"[WEB] DuckDuckGo\n" + "-" * 70); print(response)
        chat_history.add_message(HumanMessage(content=question))
        chat_history.add_message(AIMessage(content=response))
        return response

    print("Aucune information trouvee.")
    return "Aucune information trouvee."

print("Fonction ask() prete")

---
## Tests

In [ ]:
# Test 1 : DIRECT multi-articles
ask("Donne-moi les articles 5 et 8")

In [ ]:
# Test 2 : DIRECT+LLM (resume demande)
ask("Resume l'article 5")

In [ ]:
# Test 3 : RAG (question semantique)
ask("Je recrute par IA, suis-je conforme ?")

---
## Test WEB + memoire

In [ ]:
# Test 4 : DIRECT annexe
ask("Donne l'annexe III")

---
## Verifier la memoire

In [ ]:
# Afficher l'historique
print(f"Memoire : {len(chat_history.messages)} messages\n")
for m in chat_history.messages:
    role = "USER" if isinstance(m, HumanMessage) else "AI"
    print(f"  [{role:4s}] {m.content[:80]}")
    print()

---
## Mode interactif
Tapez vos questions en boucle. Entrez `quit` pour arreter.

In [ ]:
print("Agent AI Act — Tapez 'quit' pour quitter")
print("=" * 50)

while True:
    question = input("\nVotre question : ")
    if question.strip().lower() in ("quit", "exit", "q"):
        print("Au revoir !")
        break
    if not question.strip():
        continue
    ask(question)

Agent AI Act — Tapez 'quit' pour quitter
Question : date d'entrée en vigueur de l'IA Act?
